# Étape 4 : Questionnaire Patient — Prédiction COVID-19

Ce notebook permet à un patient de répondre à des **questions simples** sur son profil clinique et d'obtenir une **prédiction de son statut COVID** grâce au modèle entraîné à l'étape 3.

**Fonctionnement :**
1. Le modèle sauvegardé (le meilleur parmi les 6 testés) est chargé automatiquement.
2. Le patient répond à 13 questions simples (Oui/Non) + son âge.
3. Les réponses sont transformées au même format que les données d'entraînement.
4. Le modèle prédit le statut COVID avec une **probabilité**.

**⚠️ Avertissement médical :**
Ce questionnaire est un outil **pédagogique** développé dans le cadre d'un projet académique. Il ne remplace en aucun cas un test PCR, un test antigénique ou un avis médical professionnel. Les données d'entraînement proviennent du Mexique (2020) et peuvent ne pas refléter la situation sanitaire actuelle.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os

# ── Chargement du modèle sauvegardé ───────────────────────────────────────
MODEL_DIR = '../../data/layer_gold_data_model/modele_final'

model_path = f'{MODEL_DIR}/meilleur_modele.joblib'
meta_path = f'{MODEL_DIR}/metadata.json'

if not os.path.exists(model_path):
    print('ERREUR : Le modèle n\'a pas été trouvé.')
    print('Veuillez d\'abord exécuter le notebook prediction_covid.ipynb (Étape 3).')
else:
    modele = joblib.load(model_path)
    with open(meta_path, 'r', encoding='utf-8') as f:
        metadata = json.load(f)
    
    print(f'Modèle chargé avec succès !')
    print(f'  Algorithme : {metadata["nom_modele"]}')
    print(f'  F1-Score   : {metadata["metriques"]["f1_score"]:.4f}')
    print(f'  AUC-ROC    : {metadata["metriques"]["auc_roc"]:.4f}')
    print(f'  Features   : {len(metadata["features"])} variables')

## Comment fonctionne le questionnaire ?

Le questionnaire pose des questions correspondant exactement aux **17 variables** utilisées pour entraîner le modèle. Les réponses du patient sont transformées en un vecteur binaire (0/1) identique au format du CSV Gold.

### Transformation de l'âge
L'âge est la seule variable continue. Comme à l'étape 2, on le convertit en **4 tranches** via One-Hot Encoding :
- **Enfant** : 0-17 ans
- **Jeune** : 18-39 ans
- **Adulte** : 40-59 ans
- **Senior** : 60+ ans

### Encodage des réponses
- **Oui** → `1`
- **Non** → `0`
- **Sexe** : Femme → `0`, Homme → `1`

In [ ]:
# ── Fonctions utilitaires ─────────────────────────────────────────────────

def poser_question_oui_non(question):
    """Pose une question Oui/Non et retourne 1 (Oui) ou 0 (Non)."""
    while True:
        reponse = input(f'  {question} (oui/non) : ').strip().lower()
        if reponse in ['oui', 'o', 'yes', 'y', '1']:
            return 1
        elif reponse in ['non', 'n', 'no', '0']:
            return 0
        else:
            print('    Veuillez répondre par oui ou non.')

def poser_question_sexe():
    """Demande le sexe du patient. Retourne 0 (Femme) ou 1 (Homme)."""
    while True:
        reponse = input('  Quel est votre sexe ? (F/H) : ').strip().upper()
        if reponse in ['F', 'FEMME', '0']:
            return 0
        elif reponse in ['H', 'HOMME', 'M', '1']:
            return 1
        else:
            print('    Veuillez répondre par F (Femme) ou H (Homme).')

def poser_question_age():
    """Demande l'âge du patient et retourne les 4 variables One-Hot."""
    while True:
        try:
            age = int(input('  Quel est votre âge ? : ').strip())
            if 0 <= age <= 150:
                # One-Hot Encoding des tranches d'âge
                enfant = 1 if age <= 17 else 0
                jeune  = 1 if 18 <= age <= 39 else 0
                adulte = 1 if 40 <= age <= 59 else 0
                senior = 1 if age >= 60 else 0
                return enfant, jeune, adulte, senior
            else:
                print('    Veuillez entrer un âge valide (0-150).')
        except ValueError:
            print('    Veuillez entrer un nombre entier.')

def afficher_resultat(probabilite, seuil=0.5):
    """Affiche le résultat de la prédiction de manière claire."""
    prediction = 'POSITIF (COVID+)' if probabilite >= seuil else 'NEGATIF (COVID-)'
    couleur = 'rouge' if probabilite >= seuil else 'vert'
    
    print(f'\n{"=" * 55}')
    print(f'{" " * 10}RESULTAT DE LA PREDICTION')
    print(f'{"=" * 55}')
    print(f'\n  Probabilite COVID+ : {probabilite*100:.1f}%')
    print(f'  Prediction         : {prediction}')
    print(f'\n{"─" * 55}')
    
    # Barre visuelle de probabilité
    barre_len = 40
    rempli = int(probabilite * barre_len)
    barre = '#' * rempli + '-' * (barre_len - rempli)
    print(f'  [COVID-] |{barre}| [COVID+]')
    print(f'           0%{" " * 16}50%{" " * 15}100%')
    
    print(f'\n{"─" * 55}')
    print(f'  Ce resultat est base sur le modele {metadata["nom_modele"]}')
    print(f'  (F1-Score = {metadata["metriques"]["f1_score"]:.4f})')
    print(f'\n  Ce questionnaire est un outil pedagogique.')
    print(f'  Il ne remplace PAS un test PCR ou un avis medical.')
    print(f'{"=" * 55}')

print('Fonctions chargees.')

## Lancer le questionnaire

Exécutez la cellule ci-dessous pour démarrer le questionnaire. Répondez aux 14 questions, puis le modèle vous donnera sa prédiction avec la probabilité associée.

**Les questions portent sur :**
1. Votre sexe
2. Si vous avez été hospitalisé
3. Si vous avez une pneumonie
4. Votre âge
5-13. Vos comorbidités (diabète, BPCO, asthme, etc.)
14. Si vous fumez

In [ ]:
# ── QUESTIONNAIRE PATIENT ────────────────────────────────────────────────
print('=' * 55)
print('   QUESTIONNAIRE DE PREDICTION COVID-19')
print('=' * 55)
print('\nRepondez aux questions suivantes concernant')
print('votre profil clinique.\n')

# ── Questions ──
sexe = poser_question_sexe()
type_patient = poser_question_oui_non('Avez-vous ete hospitalise ?')
pneumonie = poser_question_oui_non('Avez-vous une pneumonie ?')

# Âge → 4 tranches
age_enfant, age_jeune, age_adulte, age_senior = poser_question_age()

# Comorbidités
print('\n  --- Comorbidites ---')
diabete = poser_question_oui_non('Etes-vous diabetique ?')
bpco = poser_question_oui_non('Avez-vous une BPCO (bronchopneumopathie chronique obstructive) ?')
asthme = poser_question_oui_non('Avez-vous de l\'asthme ?')
immunosuppression = poser_question_oui_non('Etes-vous immunodeprime(e) ?')
hypertension = poser_question_oui_non('Avez-vous de l\'hypertension ?')
autre_comorbidite = poser_question_oui_non('Avez-vous une autre comorbidite ?')
cardio = poser_question_oui_non('Avez-vous une maladie cardiovasculaire ?')
obesite = poser_question_oui_non('Etes-vous en situation d\'obesite ?')
insuffisance_renale = poser_question_oui_non('Avez-vous une insuffisance renale chronique ?')
tabagisme = poser_question_oui_non('Etes-vous fumeur/fumeuse ?')

# ── Construction du vecteur de features ──
# L'ORDRE DOIT CORRESPONDRE EXACTEMENT aux colonnes du dataset Gold
patient_data = pd.DataFrame([{
    'Sexe': sexe,
    'Type_de_patient': type_patient,
    'Pneumonie': pneumonie,
    'Diabète': diabete,
    'Bronchopneumopathie_chronique_obstructive': bpco,
    'Asthme': asthme,
    'Immunosuppression': immunosuppression,
    'Hypertension': hypertension,
    'Autre_comorbidité': autre_comorbidite,
    'Maladie_cardiovasculaire': cardio,
    'Obésité': obesite,
    'Insuffisance_rénale_chronique': insuffisance_renale,
    'Tabagisme': tabagisme,
    'Tranche_Age_tranche_age_enfant': age_enfant,
    'Tranche_Age_tranche_age_jeune': age_jeune,
    'Tranche_Age_tranche_age_adulte': age_adulte,
    'Tranche_Age_tranche_age_senior': age_senior
}])

# Réordonner les colonnes dans l'ordre exact du modèle
patient_data = patient_data[metadata['features']]

print(f'\nVotre profil clinique :')
for col, val in patient_data.iloc[0].items():
    desc = metadata['description_features'].get(col, '')
    statut = 'Oui' if val == 1 else 'Non'
    print(f'  {col:45s} → {statut}')

# ── Prédiction ──
proba = modele.predict_proba(patient_data)[0][1]  # Probabilité de COVID+
afficher_resultat(proba)

## Tester plusieurs profils

Pour tester le modèle sur différents profils sans relancer le questionnaire interactif, on peut aussi créer des profils manuellement. Voici quelques exemples types.

In [ ]:
# ── Exemples de profils pour tester le modèle ────────────────────────────
profils_test = {
    'Jeune en bonne sante': {
        'Sexe': 1, 'Type_de_patient': 0, 'Pneumonie': 0,
        'Diabète': 0, 'Bronchopneumopathie_chronique_obstructive': 0,
        'Asthme': 0, 'Immunosuppression': 0, 'Hypertension': 0,
        'Autre_comorbidité': 0, 'Maladie_cardiovasculaire': 0,
        'Obésité': 0, 'Insuffisance_rénale_chronique': 0, 'Tabagisme': 0,
        'Tranche_Age_tranche_age_enfant': 0, 'Tranche_Age_tranche_age_jeune': 1,
        'Tranche_Age_tranche_age_adulte': 0, 'Tranche_Age_tranche_age_senior': 0
    },
    'Senior avec comorbidites': {
        'Sexe': 0, 'Type_de_patient': 1, 'Pneumonie': 1,
        'Diabète': 1, 'Bronchopneumopathie_chronique_obstructive': 0,
        'Asthme': 0, 'Immunosuppression': 0, 'Hypertension': 1,
        'Autre_comorbidité': 0, 'Maladie_cardiovasculaire': 1,
        'Obésité': 1, 'Insuffisance_rénale_chronique': 0, 'Tabagisme': 0,
        'Tranche_Age_tranche_age_enfant': 0, 'Tranche_Age_tranche_age_jeune': 0,
        'Tranche_Age_tranche_age_adulte': 0, 'Tranche_Age_tranche_age_senior': 1
    },
    'Adulte hospitalise avec pneumonie': {
        'Sexe': 1, 'Type_de_patient': 1, 'Pneumonie': 1,
        'Diabète': 0, 'Bronchopneumopathie_chronique_obstructive': 0,
        'Asthme': 0, 'Immunosuppression': 0, 'Hypertension': 0,
        'Autre_comorbidité': 0, 'Maladie_cardiovasculaire': 0,
        'Obésité': 0, 'Insuffisance_rénale_chronique': 0, 'Tabagisme': 1,
        'Tranche_Age_tranche_age_enfant': 0, 'Tranche_Age_tranche_age_jeune': 0,
        'Tranche_Age_tranche_age_adulte': 1, 'Tranche_Age_tranche_age_senior': 0
    },
    'Enfant sans symptome': {
        'Sexe': 0, 'Type_de_patient': 0, 'Pneumonie': 0,
        'Diabète': 0, 'Bronchopneumopathie_chronique_obstructive': 0,
        'Asthme': 1, 'Immunosuppression': 0, 'Hypertension': 0,
        'Autre_comorbidité': 0, 'Maladie_cardiovasculaire': 0,
        'Obésité': 0, 'Insuffisance_rénale_chronique': 0, 'Tabagisme': 0,
        'Tranche_Age_tranche_age_enfant': 1, 'Tranche_Age_tranche_age_jeune': 0,
        'Tranche_Age_tranche_age_adulte': 0, 'Tranche_Age_tranche_age_senior': 0
    }
}

print('Predictions pour differents profils types :')
print(f'{"=" * 65}')

for nom_profil, profil in profils_test.items():
    df_profil = pd.DataFrame([profil])[metadata['features']]
    proba = modele.predict_proba(df_profil)[0][1]
    pred = 'COVID+' if proba >= 0.5 else 'COVID-'
    barre_len = 30
    rempli = int(proba * barre_len)
    barre = '#' * rempli + '-' * (barre_len - rempli)
    print(f'\n  {nom_profil}')
    print(f'    Probabilite COVID+ : {proba*100:.1f}%  |{barre}|  -> {pred}')

print(f'\n{"=" * 65}')
print('\nInterpretation :')
print('  - Un jeune en bonne sante a une faible probabilite.')
print('  - Un senior hospitalise avec pneumonie et comorbidites a une forte probabilite.')
print('  - Cela correspond aux connaissances medicales sur le COVID-19.')